# Project 2 — House Prices: Build & Evaluate

**Duration:** 5 min

## Load & Quick EDA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb

df = pd.read_csv('train.csv')
print(df['SalePrice'].describe())
# Log-transform target to reduce skew
df['SalePrice'] = np.log1p(df['SalePrice'])

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/ai-projects-beginner/bp-project2-build.ipynb)


## Feature Engineering

In [ ]:
# New features
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
df['TotalBath'] = df['FullBath'] + 0.5*df['HalfBath'] + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath']

# Fill missing numerics with median, categoricals with 'None'
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include='object').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna('None')

# One-hot encode
df = pd.get_dummies(df)

## Train XGBoost

In [ ]:
X = df.drop(columns=['SalePrice','Id'])
y = df['SalePrice']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                          subsample=0.8, colsample_bytree=0.8, random_state=42)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
print(f'Log RMSE: {rmse:.4f}')  # target < 0.12 in log space

> **💡 Tip:** ✅ Project 2 complete! A log RMSE of ~0.12 corresponds to roughly ±$15,000 on a $200k house.

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ Why do we log-transform the SalePrice target?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4374088896" value="0">
      <span>To make it negative</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4374088896" value="1">
      <span>To reduce right skew and stabilize variance</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4374088896" value="2">
      <span>To normalize to 0-1 range</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4374088896" value="3">
      <span>XGBoost requires it</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>